In [ ]:
import os

REPO_URL = "https://github.com/devlucascfarias/political-bias-sft.git"
if not os.path.isdir("/content/political-bias-sft"):
    !git clone {REPO_URL} /content/political-bias-sft
%cd /content/political-bias-sft
!pip install -q -r requirements.txt


In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-sft"
os.makedirs(DRIVE_ROOT, exist_ok=True)

from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

from src.config import load_config, assert_matching_hyperparameters

SMOKE_TEST = False
cfg_progressive = load_config("configs/progressive.yaml", smoke_test=SMOKE_TEST)
cfg_conservative = load_config("configs/conservative.yaml", smoke_test=SMOKE_TEST)
assert_matching_hyperparameters(cfg_progressive, cfg_conservative)


In [ ]:
from src.train import train_adapter

progressive_result = train_adapter("configs/progressive.yaml", smoke_test=SMOKE_TEST)


In [ ]:
from src.utils import clear_gpu_memory

clear_gpu_memory()


In [ ]:
conservative_result = train_adapter("configs/conservative.yaml", smoke_test=SMOKE_TEST)
clear_gpu_memory()


In [ ]:
from src.inference import load_eval_prompts, run_comparative_inference
from src.utils import write_jsonl

eval_prompts = load_eval_prompts("data")
adapter_dirs = {
    "progressive": "outputs/adapters/adapter_progressive",
    "conservative": "outputs/adapters/adapter_conservative",
}

responses = run_comparative_inference(
    cfg_progressive, eval_prompts, adapter_dirs, mode="deterministic", dataset_version="v1"
)
write_jsonl("outputs/responses/responses_deterministic.jsonl", responses)

for r in responses[:9]:
    print(f"[{r['model_variant']}] {r['prompt']}")
    print(r["response"])
    print("-" * 80)


In [ ]:
import shutil

shutil.copytree("outputs/adapters/adapter_progressive", f"{DRIVE_ROOT}/adapters/adapter_progressive", dirs_exist_ok=True)
shutil.copytree("outputs/adapters/adapter_conservative", f"{DRIVE_ROOT}/adapters/adapter_conservative", dirs_exist_ok=True)
shutil.copy("outputs/responses/responses_deterministic.jsonl", f"{DRIVE_ROOT}/responses_deterministic.jsonl")
print("Adapters e respostas salvos em", DRIVE_ROOT)
